In [1]:
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter, find_peaks, peak_widths
import matplotlib.pyplot as plt
import json
# import simplekml
import math
import matplotlib.dates as mdates
import ee
# ee.Authenticate()
ee.Initialize(project="ee-joshisur231")
import geemap
Map = geemap.Map()
import plotly.graph_objects as go
import ast
# import geopandas as gpd

In [15]:
df = pd.read_csv(r"outputs\phenology_verified_samples\ndvi_data_for_verification\stable_crop_withNDVI_glad_arcgis_used_this_in_final_crop_verification.csv").rename(columns={"CID": "point_id"})
df["date"] = pd.to_datetime(df["time"], unit= "ms")
df = df.drop(columns=["time"])

In [16]:
stable_cropland_ids = []
discarded_ids = []
ndvi_dict = {
    "point_id": [],
    # "lc": [],
    "mean": [],
    "min": [],
    "max": [],
    "std": [],
    "status": [],                    # Added to track final decision
    "total_non_crop": [],            # Added to track phenology failures
    "max_consecutive_non_crop": [],  # Added to track consecutive failures
    "total_peaks": [],                # Added to track total peaks detected
    "geo": [] 
}

grouped_points = df.sort_values(by="point_id").groupby("point_id")

for point_id, group in grouped_points:
    ts_raw = group.set_index('date')['NDVI'].resample("16D").mean()

    total_valid_obs = ts_raw.count()

    if total_valid_obs < 200:
        # print(f"Insufficient Samples for {point_id}: {total_valid_obs}")
        discarded_ids.append(point_id)

        ndvi_dict["point_id"].append(point_id)
        # ndvi_dict["lc"].append(group.iloc[0]["lc"])
        ndvi_dict["mean"].append(np.nan)
        ndvi_dict["min"].append(np.nan)
        ndvi_dict["max"].append(np.nan)
        ndvi_dict["std"].append(np.nan)
        ndvi_dict["status"].append("DISCARDED (Insufficient Samples)")
        ndvi_dict["total_non_crop"].append(np.nan)
        ndvi_dict["max_consecutive_non_crop"].append(np.nan)
        ndvi_dict["total_peaks"].append(np.nan)
        ndvi_dict["geo"].append(group.iloc[0][".geo"])
        continue

    #Interpolate values across the entire 23 years
    ts_interp = ts_raw.interpolate(method="linear").bfill().ffill()

    # half_window = math.floor(year_valid_obs / 4) #timesat logic
    window = 11 #(2 * half_window) + 1
    
    smoothed_ndvi = savgol_filter(ts_interp.values, window_length = window, polyorder=2)
    smoothed_ndvi_series = pd.Series(smoothed_ndvi, index=ts_interp.index)

    peaks, properties = find_peaks(
        smoothed_ndvi, 
        height=0.35,         # Kept the same: reliably identifies active vegetation
        distance= 5,          # Lowered from 7: Allows peaks to be 80 days apart (Captures tight multi-cropping)
        width=(5, 15),       # Lowered from 7: 7 samples (80 days) captures the rapid Terai cycles, 15 caps the forests
        prominence=0.10      # Lowered from 0.12: Accounts for overlapping crop cycles where NDVI doesn't hit bare soil
    )
   
    peak_dates = ts_interp.index[peaks]

    yearly_classification = {}
    yearly_peaks = []

    for year in range(2000, 2023):
        smoothed_year = smoothed_ndvi_series[smoothed_ndvi_series.index.year == year]

        if len(smoothed_year) == 0:
            yearly_classification[year] = "insufficient_data"
            continue

        peaks_in_year = (peak_dates.year == year).sum()
        yearly_peaks.append(smoothed_year.max())

        if peaks_in_year >=1:
            yearly_classification[year] = "crop"
        else:
            yearly_classification[year] = "non_crop"
    
    overall_mean = smoothed_ndvi_series.mean()
    overall_min =  smoothed_ndvi_series.min()
    overall_max = smoothed_ndvi_series.max()
    overall_amplitude = overall_max - overall_min
    std_max = np.std(yearly_peaks, ddof=1) if len(yearly_peaks) > 0 else 1.0

    total_non_crop = 0
    consecutive_non_crop = 0
    max_consecutive_non_crop = 0

    for year in range(2000, 2023):
        if yearly_classification.get(year) == "non_crop":
            total_non_crop += 1
            consecutive_non_crop += 1
            if consecutive_non_crop > max_consecutive_non_crop:
                max_consecutive_non_crop = consecutive_non_crop
        else:
            consecutive_non_crop = 0

    # NEW: Discard if it fails phenology OR if it lacks the massive swing of a harvested crop
    if total_non_crop >= 4 or max_consecutive_non_crop >= 3 or overall_amplitude < 0.55 or overall_min > 0.25: 
        discarded_ids.append(point_id)
        if overall_amplitude < 0.55 or overall_min > 0.25:
            status_reason = "DISCARDED (Static Vegetation / Low Amplitude)"
        else:
            status_reason = "DISCARDED (Failed Phenology)"
    else:
        stable_cropland_ids.append(point_id)
        status_reason = "STABLE CROPLAND"
    
    ndvi_dict["point_id"].append(point_id)
    # ndvi_dict["lc"].append(group.iloc[0]["lc"])
    ndvi_dict["mean"].append(round(overall_mean, 4))
    ndvi_dict["min"].append(round(overall_min, 4))
    ndvi_dict["max"].append(round(overall_max, 4))
    ndvi_dict["std"].append(round(std_max, 4))
    ndvi_dict["status"].append(status_reason)
    ndvi_dict["total_non_crop"].append(total_non_crop)
    ndvi_dict["max_consecutive_non_crop"].append(max_consecutive_non_crop)
    ndvi_dict["total_peaks"].append(len(peaks))
    ndvi_dict["geo"].append(group.iloc[0][".geo"])
    # fig = go.Figure()
    # # 2. Raw Resampled (Scatter Dots)
    # fig.add_trace(go.Scatter(
    #     x=ts_raw.index, 
    #     y=ts_raw.values,
    #     mode='markers',
    #     name='Raw Resampled',
    #     marker=dict(color='grey', opacity=0.5),
    #     showlegend=True
    # ))

    # # 3. Raw Resampled (Dashed Connecting Line)
    # fig.add_trace(go.Scatter(
    #     x=ts_raw.index, 
    #     y=ts_raw.values,
    #     mode='lines',
    #     name='Raw Resampled (Line)',
    #     line=dict(color='black', width=1, dash='dash'),
    #     opacity=0.6,
    #     showlegend=False # Matplotlib showed 1 legend item for both, this keeps it clean
    # ))

    # # 4. Savgol Smoothed (Solid Green Line)
    # fig.add_trace(go.Scatter(
    #     x=ts_interp.index, 
    #     y=smoothed_ndvi,
    #     mode='lines',
    #     name='Savgol Smoothed',
    #     line=dict(color='green', width=1.5),
    #     opacity=0.7
    # ))

    # # 5. Detected Peaks (Red X Markers)
    # if len(peaks) > 0:
    #     fig.add_trace(go.Scatter(
    #         x=ts_interp.index[peaks], 
    #         y=smoothed_ndvi[peaks],
    #         mode='markers',
    #         name='Detected Peaks',
    #         marker=dict(symbol='x', color='red', size=10, line=dict(width=2, color='red'))
    #     ))


    # info_text = (
    #     f"Class: {group.iloc[0]['lc']}<br>"
    #     f"Mean, Min, Max: {round(overall_mean, 2)}, {round(overall_min, 2)}, {round(overall_max, 2)}<br>"
    #     f"Peak Std Dev: {std_max:.3f}<br>"
    #     f"Final Status: <b>{status_reason}</b><br>"
    #     f"Total Non-Crop Years: {total_non_crop}<br>"
    #     f"Max Consecutive Non-Crop: {max_consecutive_non_crop}<br>"
    #     f"Total Peaks (2000-2022): {len(peaks)}"
    # )

    # ndvi_dict["point_id"].append(point_id)
    # ndvi_dict["lc"].append(group.iloc[0]["lc"])
    # ndvi_dict["mean"].append(overall_mean)
    # ndvi_dict["min"].append(overall_min)
    # ndvi_dict["max"].append(overall_max)
    # ndvi_dict["std"].append(std_max)

    # fig.add_annotation(
    #     x=0.7,
    #     y=1.4,
    #     xref="paper",
    #     yref="paper",
    #     text=info_text,
    #     showarrow=False,
    #     align="left",
    #     bgcolor="rgba(255, 255, 255, 0.8)", # White background with 80% opacity
    #     bordercolor="gray",
    #     borderwidth=1,
    #     borderpad=6,
    #     font=dict(size=11),
    #     xanchor="left",
    #     yanchor="top"
    # )

    # # 7. Layout formatting (Mimicking Matplotlib styling)
    # fig.update_layout(
    #     title=f"Continuous 23-Year Phenology Profile",
    #     height=450,           # Similar to figsize=(..., 5)
    #     plot_bgcolor='white', # Removes Plotly's default gray background
        
    #     # Y-Axis Settings (-0.1 to 1.0)
    #     yaxis=dict(
    #         range=[-0.1, 1.0], 
    #         gridcolor='rgba(128,128,128,0.2)', # Faint grid lines
    #         zerolinecolor='rgba(128,128,128,0.5)'
    #     ),
        
    #     # X-Axis Settings (1-year ticks, 45 degree angle)
    #     xaxis=dict(
    #         dtick="M12",          # Major ticks every 12 Months
    #         tickformat="%Y",      # Show only the Year
    #         tickangle=45,
    #         gridcolor='rgba(128,128,128,0.2)'
    #     ),
        
    #     # Legend position (Lower Right)
    #     legend=dict(
    #         x=0.99,
    #         y=0.02,
    #         xanchor="right",
    #         yanchor="bottom",
    #         bgcolor="rgba(255,255,255,0.7)"
    #     )
    # )

    # fig.show()

    
print(f"Total points analyzed: {len(grouped_points)}")
print(f"Pure Cropland Points Kept: {len(stable_cropland_ids)}")
print(f"Points Discarded: {len(discarded_ids)}")

results_df = pd.DataFrame(ndvi_dict)

Total points analyzed: 5522
Pure Cropland Points Kept: 1898
Points Discarded: 3624


In [17]:
results_df.value_counts("status")

status
DISCARDED (Insufficient Samples)                 2158
STABLE CROPLAND                                  1898
DISCARDED (Failed Phenology)                     1092
DISCARDED (Static Vegetation / Low Amplitude)     374
Name: count, dtype: int64

In [10]:
results_df.value_counts("status")

status
DISCARDED (Insufficient Samples)                 1611
DISCARDED (Static Vegetation / Low Amplitude)    1413
STABLE CROPLAND                                   332
DISCARDED (Failed Phenology)                      255
Name: count, dtype: int64

In [7]:
stable_crop = results_df[results_df["status"] == "STABLE CROPLAND"]
stable_crop["coords"] = stable_crop["geo"].apply(lambda x: ast.literal_eval(x)["coordinates"])
stable_crop["x"] = stable_crop["coords"].apply(lambda x: x[0])
stable_crop["y"] = stable_crop["coords"].apply(lambda x: x[1])
stable_crop.to_csv(r"test2.csv")

In [8]:
stable_crop.shape

(332, 13)

In [12]:
results_df.to_csv(r"outputs\phenology_verified_samples\stable_ag_phenology_classification_results_glad.csv", index=False)

## Prepare for manual validation

In [12]:
import pandas as pd
import ee
ee.Initialize(project = "ee-joshisur231")
import json
# import simplekml
import geopandas as gpd
import geemap
Map = geemap.Map()
import ast

In [2]:
final_stable_crop = pd.read_csv(r"outputs\phenology_verified_samples\stable_ag_phenology_classification_results_glad.csv")
final_stable_crop["coords"] = final_stable_crop["geo"].apply(lambda x: ast.literal_eval(x)["coordinates"])
final_stable_crop["x"] = final_stable_crop["coords"].apply(lambda x: x[0])
final_stable_crop["y"] = final_stable_crop["coords"].apply(lambda x: x[1])
final_stable_crop =  final_stable_crop[final_stable_crop["status"] == "STABLE CROPLAND"]

In [8]:
ee_final_stable_crop = geemap.df_to_ee(
    final_stable_crop,
    latitude = "y",
    longitude = "x" 
)

geo_region = ee.Image("projects/ee-joshisur231/assets/pa_effectiveness/geoReg_nepal").rename("reg")
dem = ee.Image("USGS/SRTMGL1_003")
slope = ee.Terrain.slope(dem)

ee_final_stable_crop_withGeo = geo_region.reduceRegions(collection = ee_final_stable_crop, scale = 30, crs="EPSG:32645", reducer=ee.Reducer.first()).map(lambda feat: feat.set("geoReg", feat.get("first")).select(['point_id', 'mean', 'min', 'max', 'std', 'status', 'total_non_crop','max_consecutive_non_crop', 'total_peaks', 'x', 'y', 'geoReg']))
# ee_final_stable_crop_withGeo_slope = slope.reduceRegions(collection = ee_final_stable_crop_withGeo, scale = 30, reducer=ee.Reducer.first()).map(lambda feat: feat.set("slope", feat.get("first")).select(['point_id', 'mean', 'min', 'max', 'std', 'status', 'total_non_crop','max_consecutive_non_crop', 'total_peaks', 'orig_x', 'orig_y', 'geoReg', 'slope']))

final_stable_crop_withGeo = geemap.ee_to_df(ee_final_stable_crop_withGeo, remove_geom=False)
# removing points on very high slope
# final_stable_crop_withGeo_slope = final_stable_crop_withGeo_slope[final_stable_crop_withGeo_slope["slope"] < 35].dropna()

In [9]:
samples_size_geoReg = {1:20, 2:20, 3: 180} #sample size each zone (10% of stable divided based on area)

stable_valid_samples = final_stable_crop_withGeo.groupby("geoReg", group_keys=True).apply(lambda x: x.sample(n=samples_size_geoReg.get(x.name, 0))).reset_index()

In [10]:
stable_valid_samples[['geoReg', 'x','y',
'mean', 'min','max',  'std',  'max_consecutive_non_crop', 'total_non_crop','total_peaks','status',
  'point_id']].to_csv(r"outputs\phenology_verified_for_validation\final_stable_ag_manual_validation_samples_glad.csv")

In [ ]:
# gdf = gpd.GeoDataFrame(stable_valid_samples[['geoReg', 'slope', 'orig_x','orig_y',
# 'mean', 'min','max',  'std',  'max_consecutive_non_crop', 'total_non_crop','total_peaks','status',
#   'point_id']], geometry= gpd.points_from_xy(stable_valid_samples.orig_x, stable_valid_samples.orig_y), crs="EPSG:4326")
# gdf.to_file(r"outputs/spatial_data/new/stable_crop_for_manual_validation_new.shp", driver="ESRI Shapefile")

In [14]:
gdf = gpd.read_file(r"outputs\phenology_verified_for_validation\final_stable_ag_manual_validation_samples_glad.csv")
gdf = gpd.GeoDataFrame(gdf, geometry=gpd.points_from_xy(gdf.x, gdf.y),crs="EPSG:4326")
gdf = gdf.to_crs(epsg="32645")
gdf_buffer = gdf.buffer(15, cap_style=3)

In [15]:
gdf_buffer.to_file(r"outputs/spatial_data/new/stable_crop_for_manual_validation_grids_glad.shp", driver="ESRI Shapefile")

## Validation

In [1]:
from helpers import config
import pandas as pd
from sklearn import metrics
import matplotlib.pyplot as plt
from sklearn import metrics

loaded config!


In [3]:
df = pd.read_csv(r"outputs\phenology_verified_for_validation\final_stable_ag_manual_validation_samples_glad.csv")

y_true = df["actual"]
y_pred = df["predicted"]
print("Final validation report")
conf_matrix = pd.crosstab(
    y_true, 
    y_pred, 
    rownames=['Actual'], 
    colnames=['Predicted']
)
print(conf_matrix)

print("\n=== CLASSIFICATION REPORT ===")
print(metrics.classification_report(
    y_true, 
    y_pred, 
    target_names=['non crop', 'Stable Cropland']
))

Final validation report
Predicted    1
Actual        
0           23
1          197

=== CLASSIFICATION REPORT ===
                 precision    recall  f1-score   support

       non crop       0.00      0.00      0.00        23
Stable Cropland       0.90      1.00      0.94       197

       accuracy                           0.90       220
      macro avg       0.45      0.50      0.47       220
   weighted avg       0.80      0.90      0.85       220



e:\Software\conda_envs\gee\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Software\conda_envs\gee\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Software\conda_envs\gee\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
